# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access dataset metadata as a Python object
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}\n")
print(f"Published: {metadata.datePublished}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets, fields, and columns by their @id

record_sets = []

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for record_set in metadata.recordSet:
        # Each record_set should have an @id, name, and fields
        rs_id = getattr(record_set, '@id', None)
        rs_name = getattr(record_set, 'name', None)
        print(f"Record set: {rs_name} (@id: {rs_id})")
        record_sets.append(rs_id)
        # Fields in this record set
        if hasattr(record_set, 'field'):
            for field in getattr(record_set, 'field'):
                f_id = getattr(field, '@id', None)
                f_name = getattr(field, 'name', None)
                print(f"    Field: {f_name} (@id: {f_id})  Datatype: {getattr(field, 'dataType', None)}")
                # If columns exist
                if hasattr(field, 'column'):
                    for column in getattr(field, 'column'):
                        c_id = getattr(column, '@id', None)
                        c_name = getattr(column, 'name', None)
                        print(f"        Column: {c_name} (@id: {c_id})")
else:
    print("No record sets found in metadata. Please check the Croissant file.")

# If no record sets found, try the next code cell for demo purposes
if len(record_sets) == 0:
    # Try loading and iterating available record sets by introspecting dataset.records()
    print("Attempting to retrieve record sets dynamically...")
    try:
        # mlcroissant from v1.0.0: expose_record_sets()
        available_sets = dataset.expose_record_sets()
        print(f"Available record sets: {available_sets}")
        record_sets.extend(available_sets)
    except Exception as e:
        print("Could not dynamically retrieve record sets.", e)

# If still missing, manually set record set @id for downstream cells
record_set_id = record_sets[0] if len(record_sets) > 0 else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

_Note: All Croissant entities are referenced by their `@id`._

In [ ]:
# If the previous cell didn't find record sets, set an example @id (KNOWN from the FAIR² Croissant file analysis)
if not record_set_id:
    # Example: In FAIR², the main table is usually named 'dv:dataset' or similar
    record_set_id = 'dv:dataset'

print(f"Selected record set for extraction: {record_set_id}")

# Extract all records from the selected record set
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)

# Preview the available columns and the first few rows
print(f"Columns in record set {record_set_id}:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering records, normalizing numeric fields, and grouping data.
All processing references fields by their `@id`.

In [ ]:
# Choose a numeric field (by @id) for demonstration.
# Common quantitation fields in mass spec datasets: e.g., 'cr:mw', 'cr:pi', 'cr:abundance', etc.
# Let's select one that appears typical. If not found, replace with an actual column shown in above output.

candidate_numeric_fields = ['cr:mw', 'cr:pi', 'cr:abundance', 'cr:peptideCount']
numeric_field_id = None
for candidate in candidate_numeric_fields:
    if candidate in df.columns:
        numeric_field_id = candidate
        break
if not numeric_field_id:
    # Fallback: use the first numeric-looking column
    numeric_columns = df.select_dtypes(include=['number']).columns
    if len(numeric_columns) > 0:
        numeric_field_id = numeric_columns[0]

print(f"Selected numeric field for EDA: {numeric_field_id}")

if numeric_field_id:
    # Drop missing values for fair analysis
    df_clean = df.dropna(subset=[numeric_field_id]).copy()

    threshold = df_clean[numeric_field_id].mean()  # Example: filter values above the mean
    filtered_df = df_clean[df_clean[numeric_field_id] > threshold]
    print(f"Number of records with {numeric_field_id} > {threshold:.3f}: {len(filtered_df)}")
    print(filtered_df[[numeric_field_id]].head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"First 5 normalized values for {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Try grouping by a categorical attribute (e.g., 'cr:accession', 'cr:sample', 'cr:group'). Pick one present in df.
    group_field_candidates = ['cr:sample', 'cr:condition', 'cr:experiment', 'cr:accession']
    group_field = None
    for col in group_field_candidates:
        if col in df.columns:
            group_field = col
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA. Please review available columns.")

## 5. Visualization
Visualize distributions and relationships in the dataset.

_Note: Visualizations reference field `@id`s as axis labels._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the selected numeric field
if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If there's a grouping field, boxplot
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a mass spectrometry-based protein dataset using the `mlcroissant` library. Using Croissant `@id` references, we examined the record sets, extracted tabular data, performed basic filtering, normalization, and grouping, and visualized the main quantitative field. This approach is robust for FAIR data packages in life sciences and can be easily adapted to other Croissant datasets.

For deeper biological interpretation, see [the dataset source](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and its formal documentation.